# PID-RL PoC — Continue Training from HF

Continúa el entrenamiento del modelo existente en HuggingFace para más steps.

- **Modelo base**: `Fenryr/qwen2.5-1.5B-pid-v1` (50 steps previos)
- **Target**: 100-200 steps adicionales
- **Repo original**: https://github.com/Grinta-Protocol/qwen-rl-shanghai

**Pipeline:** HF LoRA → cargar → GRPO continue → descargar → evaluar

## 1 · Environment check

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## 2 · Install dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/huggingface/trl.git
!pip install yfinance

## 3 · Clone repo + login a HF

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/qwen-rl-shanghai"
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone",
        "https://github.com/Grinta-Protocol/qwen-rl-shanghai.git",
        REPO_DIR,
    ], check=True)

sys.path.insert(0, f"{REPO_DIR}/pid_rl")
os.chdir(f"{REPO_DIR}/pid_rl")
!ls

In [ ]:
# Login a HuggingFace (necesario para descargar modelo privado o pushear)
# Descomenta y pon tu token si vas a pushear
# from huggingface_hub import login
# login("TU_TOKEN_AQUI")  # token con write access

## 4 · Imports

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import copy
import numpy as np
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from config import TRAIN, SIM, GUARD, PID
from sim import Simulator
from scenarios import generate_batch
from prompt import build_prompt, parse_output, ParseError
from reward import compute_reward
from baselines import StaticGainsPolicy, HeuristicPolicy, RandomPolicy

print(f"Base model: {TRAIN.base_model}")
print(f"LoRA rank:  {TRAIN.lora_rank}")
print(f"bf16:       {is_bfloat16_supported()}")

## 5 · Load base model + LoRA desde HF

**KEY DIFFERENCE**: Aquí CARGAMOS el LoRA existente desde HF antes de entrenar.
Esto permite continuar el training desde donde quedó (en vez de empezar de cero).

In [ ]:
MAX_SEQ_LEN = TRAIN.max_prompt_length + TRAIN.max_completion_length

# LIMPIAR MEMORIA primero
import gc
gc.collect()
torch.cuda.empty_cache()

# Cargar TODO en 4-bit para ahorrar VRAM
model_name = "unsloth/Qwen2.5-1.5B-Instruct"
adapter_name = "Fenryr/qwen2.5-1.5B-pid-v1"

print(f"Loading model + adapter in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,  # ← 4-bit para ahorrar
    fast_inference=False,
    max_lora_rank=TRAIN.lora_rank,
    adapter_name=adapter_name,
)

print("✓ Model + LoRA loaded from HF")
print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6 · Build training dataset

Mismo proceso que notebook original — genera scenarios para training.

In [ ]:
N_TRAIN_SCENARIOS = 256
PRE_ROLL_MIN, PRE_ROLL_MAX = 10, 80

scenarios_pool = generate_batch(n=N_TRAIN_SCENARIOS, seed=TRAIN.seed)

sim_pool = []
prompt_messages_pool = []
rng = np.random.default_rng(seed=TRAIN.seed + 1)

for idx, scenario in enumerate(scenarios_pool):
    sim = Simulator(
        btc_path=scenario.btc_path,
        initial_kp=scenario.initial_kp,
        initial_ki=scenario.initial_ki,
        rng=np.random.default_rng(seed=TRAIN.seed + idx),
    )
    pre_roll = int(rng.integers(PRE_ROLL_MIN, PRE_ROLL_MAX))
    history = sim.run_forward(n_steps=pre_roll)
    state = sim.get_state()
    dev_hist = [h.deviation_pct for h in history[-8:]]
    sys_p, user_p = build_prompt(
        state=state,
        btc_history=scenario.btc_path[: sim.step_idx + 1],
        deviation_history=dev_hist,
    )
    sim_pool.append(sim)
    prompt_messages_pool.append([
        {"role": "system", "content": sys_p},
        {"role": "user",   "content": user_p},
    ])

dataset = Dataset.from_list([
        {"prompt": prompt_messages_pool[i], "scenario_idx": i}
        for i in range(len(scenarios_pool))
])
print(dataset)
print(f"Generated {len(scenarios_pool)} training scenarios")

## 7 · Reward functions

In [ ]:
def _completion_text(c):
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return "\n".join(str(m.get("content", "")) for m in c)
    return str(c)


def pid_reward(prompts, completions, scenario_idx, **kwargs):
    rewards = []
    for completion, idx in zip(completions, scenario_idx):
        text = _completion_text(completion)
        sim = copy.deepcopy(sim_pool[idx])
        br = compute_reward(text, sim)
        rewards.append(float(br.total))
    return rewards


def json_validity(prompts, completions, **kwargs):
    """Small shaping: +0.5 if the output parses, -0.5 otherwise."""
    rewards = []
    for completion in completions:
        text = _completion_text(completion)
        parsed = parse_output(text)
        rewards.append(0.5 if not isinstance(parsed, ParseError) else -0.5)
    return rewards


# Smoke test
sample = '{"action":"adjust","new_kp":2.5,"new_ki":0.002,"is_emergency":false,"reasoning":"test"}'
print("pid_reward sample:",   pid_reward([None], [sample], [0]))
print("json_validity sample:", json_validity([None], [sample]))

## 8 · GRPO config — **100 o 200 STEPS**

In [ ]:
# CONTINUAR TRAINING: 100 o 200 steps adicionales
CONTINUE_STEPS = 200  # CAMBIA ESTO: 100 o 200

training_args = GRPOConfig(
    output_dir="outputs_continue",
    use_vllm=False,
    learning_rate=TRAIN.learning_rate,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=TRAIN.warmup_ratio,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=10,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=TRAIN.batch_size,
    gradient_accumulation_steps=1,
    num_generations=TRAIN.num_generations,
    max_steps=CONTINUE_STEPS,  # ← 100 o 200
    save_steps=50,
    max_grad_norm=0.1,
    beta=TRAIN.kl_coef,
    seed=TRAIN.seed,
    report_to="none",
)
print(f"Training config: {CONTINUE_STEPS} additional steps")
print(f"Learning rate: {TRAIN.learning_rate}")
print(f"KL coef: {TRAIN.kl_coef}")

## 9 · Train — Continue from existing LoRA

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[pid_reward, json_validity],
    args=training_args,
    train_dataset=dataset,
)

print("Starting continued training...")
trainer.train()
print("✓ Training complete!")

## 10 · Save LoRA + Download

In [ ]:
# Guardar LoRA
output_name = f"pid_rl_lora_v1_{CONTINUE_STEPS}step"
model.save_pretrained(output_name)
tokenizer.save_pretrained(output_name)

!ls -la {output_name}

print(f"\n✓ LoRA saved to ./{output_name}")
print("Download from Files panel or:")

In [ ]:
# Download directo desde Colab
from google.colab import files
import shutil

# Zip para descargar
shutil.make_archive(output_name, 'zip', output_name)
files.download(f"{output_name}.zip")
print(f"Downloaded {output_name}.zip")

## 11 · (Opcional) Push to HF Hub

In [ ]:
# Descomenta para pushear a HF
# model.push_to_hub_merged(
#     "TU_USUARIO/qwen2.5-1.5B-pid-v2",  # Nuevo nombre
#     tokenizer,
#     save_method="lora",
#     token="TU_TOKEN"
# )

## 12 · Evaluate vs baselines

In [ ]:
from eval import run_full_eval


class TrainedModelPolicy:
    """Wraps the trained model to match the baselines.Policy interface."""
    name = "trained"

    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.3):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        FastLanguageModel.for_inference(self.model)

    def decide(self, state, btc_history, deviation_history):
        sys_p, user_p = build_prompt(
            state=state,
            btc_history=btc_history,
            deviation_history=deviation_history,
        )
        messages = [
            {"role": "system", "content": sys_p},
            {"role": "user",   "content": user_p},
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        gen = self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return gen


trained_policy = TrainedModelPolicy(model, tokenizer)

policies = [
    StaticGainsPolicy(),
    HeuristicPolicy(),
    RandomPolicy(rng=np.random.default_rng(seed=123)),
    trained_policy,
]

results = run_full_eval(
    policies=policies,
    n_synthetic_holdout=24,
    include_real=True,
    verbose=False,
)

## 13 · Results Summary

In [ ]:
# Print results nicely
import json

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

for bench_name, bench_data in results.items():
    print(f"\n### {bench_name.upper()}")
    for pol_name, pol_data in bench_data.items():
        mean_r = pol_data.get("mean_reward", "N/A")
        mean_d = pol_data.get("mean_abs_dev", "N/A")
        print(f"  {pol_name:12} | reward: {mean_r:8.2f} | |dev|: {mean_d:.2f}%")

print("\n" + "=" * 60)
print("BASELINES TO BEAT:")
print("  Synthetic: static > -89.76, heuristic > -89.68")
print("  Real:      heuristic > -159.61")
print("=" * 60)

## 14 · Qualitative peek

In [ ]:
from scenarios import _gen_crash, _gen_pump, _gen_stable

peek_rng = np.random.default_rng(seed=777)
for name, gen in [("crash", _gen_crash), ("pump", _gen_pump), ("stable", _gen_stable)]:
    sc = gen(peek_rng, SIM.episode_steps)
    sim = Simulator(
        btc_path=sc.btc_path, initial_kp=sc.initial_kp, initial_ki=sc.initial_ki,
        rng=np.random.default_rng(seed=7),
    )
    hist = sim.run_forward(n_steps=50)
    dev_hist = [h.deviation_pct for h in hist[-8:]]
    state = sim.get_state()
    completion = trained_policy.decide(
        state=state,
        btc_history=sc.btc_path[: sim.step_idx + 1],
        deviation_history=dev_hist,
    )
    parsed = parse_output(completion)
    print(f"\n=== {name} ===")
    print(f"  btc: {sc.btc_path[sim.step_idx - 5]:.0f} → {sc.btc_path[sim.step_idx]:.0f}")
    print(f"  dev: {state.deviation_pct:+.3f}%   kp={state.kp:.3f}  ki={state.ki:.5f}")
    print(f"  model: {completion[:200]}...")
    print(f"  parsed: {parsed}")